# 22 — 강력한 Face Inpainting (Phase 7-P)

**목적**: 본 시스템의 변형 약함 문제를 **face 특화 inpainting 모델 + Strength 파라미터 강제 + 강한 prompt**로 근본 해결.

**핵심 차이점 (vs 노트북 21)**:
1. **Face 특화 모델 사용** — DreamShaper, RealisticVision V6.0 등 face fine-tuned
2. **Strength 파라미터 명시** (0.95+) — 변형 강도 직접 제어
3. **다중 모델 자동 시도** — 가용 모델 중 가장 좋은 결과
4. **Negative prompt 강화** — 'same person, identical' 제거 (변형 허용)

**예상 결과**: 시술 효과 명확 + identity 유지

**예상 소요**: T4 GPU 기준 약 15-20분

## 1. 환경 셋업

In [ ]:
!nvidia-smi

In [ ]:
!pip install --quiet diffusers transformers accelerate safetensors
!pip install --quiet mediapipe albumentations pyyaml imageio

In [ ]:
import torch, time, os, sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('device:', device)

In [ ]:
# Git clone
REPO_LOCAL = '/content/cv-final'
if not os.path.exists(REPO_LOCAL):
    !git clone https://github.com/keonU206/cv-final.git {REPO_LOCAL}
else:
    !cd {REPO_LOCAL} && git pull --quiet

for m in list(sys.modules.keys()):
    if m.startswith('project'):
        del sys.modules[m]

if REPO_LOCAL not in sys.path:
    sys.path.insert(0, REPO_LOCAL)

import importlib
importlib.invalidate_caches()

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/cv-final'
OUTPUT_DIR = Path(DRIVE_ROOT) / 'data' / 'outputs' / 'strong_inpaint'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ 환경 준비 완료')
print(f'   OUTPUT_DIR: {OUTPUT_DIR}')

## 2. 사진 + Mask 준비

In [ ]:
from project.input_generator import make_mask, load_procedures

SIZE = 512  # face 특화 SD 1.5 기준

from google.colab import files
print('📷 사진 업로드:')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

image_rgb = cv2.imread(img_path)
image_rgb = cv2.cvtColor(image_rgb, cv2.COLOR_BGR2RGB)
image_rgb = cv2.resize(image_rgb, (SIZE, SIZE))

plt.figure(figsize=(6, 6))
plt.imshow(image_rgb); plt.title('원본'); plt.axis('off')
plt.show()

In [ ]:
# MediaPipe landmark
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

!wget -q -O /tmp/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='/tmp/face_landmarker.task'),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,
)
landmarker = FaceLandmarker.create_from_options(options)

mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
result = landmarker.detect(mp_image)
landmarks_2d = np.array([[lm.x * SIZE, lm.y * SIZE] for lm in result.face_landmarks[0]])
print(f'478 landmark 추출 완료')

In [ ]:
# Mask 생성 + 대폭 확장
procedures_db = load_procedures()
PROC_ID = 'nose_tip'

procedure = procedures_db[PROC_ID]
mask_raw = make_mask(landmarks_2d.astype(np.int32), procedure, SIZE)
mask = mask_raw[:, :, 0] if mask_raw.ndim == 3 else mask_raw

MASK_DILATE = {
    'nose_tip': 40,        # ⭐ 매우 크게
    'double_eyelid': 30,
    'jaw_v_line': 50,
}
ds = MASK_DILATE[PROC_ID]
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ds, ds))
mask = cv2.dilate(mask, kernel, iterations=1)

mask_pct = 100 * (mask > 0).sum() / (SIZE * SIZE)
print(f'Mask 영역: {mask_pct:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_rgb); axes[0].set_title('원본'); axes[0].axis('off')
axes[1].imshow(mask, cmap='gray'); axes[1].set_title(f'Mask (+{ds}px)'); axes[1].axis('off')
overlay = image_rgb.copy()
overlay[mask > 0] = (overlay[mask > 0] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## 3. ⭐ Face 특화 Inpainting 모델 로드

**다중 모델 자동 시도** (가장 좋은 결과):
1. **DreamShaper-8 Inpainting** (face 특화 SD 1.5)
2. **Realistic Vision V6.0 Inpainting** (포토리얼)
3. **Default SD Inpaint** (fallback)

In [ ]:
from diffusers import StableDiffusionInpaintPipeline, DPMSolverMultistepScheduler

# 강력한 face 특화 모델들 (시도 순서)
FACE_MODELS = [
    'Lykon/dreamshaper-8-inpainting',           # face 특화 1순위
    'Uminosachi/realisticVisionV60B1_v51VAE-inpainting',  # 포토리얼 face
    'Lykon/absolute-reality-1.6525-inpainting', # 포토리얼
    'runwayml/stable-diffusion-inpainting',     # 기본 (fallback)
]

pipe = None
for model_id in FACE_MODELS:
    try:
        print(f'\n📦 모델 로드 시도: {model_id}')
        pipe = StableDiffusionInpaintPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            safety_checker=None,
        ).to(device)
        print(f'✅ 로드 성공: {model_id}')
        MODEL_USED = model_id
        break
    except Exception as e:
        print(f'⚠ 실패: {str(e)[:100]}')
        continue

if pipe is None:
    raise RuntimeError('모든 모델 로드 실패')

# DPMSolver (빠른 sampler)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# Attention slicing (메모리 절감)
try:
    pipe.enable_attention_slicing()
    print('✅ Attention slicing 활성화')
except Exception:
    pass

print(f'\n✅ Pipeline 준비')
print(f'   Model: {MODEL_USED}')
print(f'   Device: {next(pipe.unet.parameters()).device}')
print(f'   GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 4. ⭐ 강력한 Prompt (변형 강조)

In [ ]:
# 강력한 prompt — identity 제약 최소, 시술 효과 극대화
STRONG_PROMPTS = {
    'nose_tip': {
        'positive': 'beautiful slim refined small nose tip after professional rhinoplasty surgery, elegant aesthetic dramatic nose shape change, perfect Korean K-beauty standard, ultra photorealistic portrait, smooth flawless skin, professional studio photography, sharp focus, ultra detailed face, dramatic transformation, beautiful refined nose',
        'negative': 'wide bulbous nose, large flat nose tip, deformed, ugly, blurry, low quality, cartoon, anime, sketch, painting, oversaturated, asymmetric, bad anatomy, watermark, text, makeup heavy, oily skin, double person',
    },
    'double_eyelid': {
        'positive': 'beautiful natural double eyelid after professional blepharoplasty surgery, dramatic eye transformation, bright clear large open eyes, elegant Korean K-beauty, ultra photorealistic, perfect aesthetic almond eyes with crease, professional portrait, detailed face, sharp focus, beautiful eyes',
        'negative': 'closed eyes, sleepy eyes, monolid, asymmetric eyes, deformed, ugly, blurry, cartoon, anime, low quality, heavy eye makeup, oversaturated, double person',
    },
    'jaw_v_line': {
        'positive': 'beautifully refined slim sharp v-line jaw after professional surgery, dramatic jaw transformation, elegantly refined narrow chin, perfect Korean K-beauty face shape, ultra photorealistic portrait, smooth flawless skin, professional studio photography, detailed face, sharp focus, beautiful jaw',
        'negative': 'wide square jaw, masculine face, double chin, fat face, deformed, ugly, blurry, low quality, cartoon, anime, heavy makeup, oversaturated, double person',
    },
}

current = STRONG_PROMPTS[PROC_ID]
print(f'시술: {PROC_ID}')
print(f'\nPositive: {current["positive"][:100]}...')
print(f'\nNegative: {current["negative"][:80]}...')

## 5. ⭐ Strength 파라미터로 변형 강도 직접 제어

- **strength=0.8**: 중간 변형 (안전)
- **strength=0.95**: 강한 변형 ⭐ 추천
- **strength=1.0**: 완전 재생성 (가장 강함)

In [ ]:
# 실험 — Strength 3가지 × Guidance 3가지 = 9개 결과
INFERENCE_STEPS = 50
STRENGTH_VALUES = [0.85, 0.95, 1.0]   # ⭐ 강한 변형
GUIDANCE_VALUES = [8.5, 10.0, 12.0]
SEED = 42

image_pil = Image.fromarray(image_rgb)
mask_pil = Image.fromarray(mask)

results = {}
total_start = time.time()

for s in STRENGTH_VALUES:
    for g in GUIDANCE_VALUES:
        key = f's{s}_g{g}'
        print(f'\n━━━ {key} ━━━')
        t0 = time.time()

        try:
            with torch.autocast('cuda'):
                output = pipe(
                    prompt=current['positive'],
                    negative_prompt=current['negative'],
                    image=image_pil,
                    mask_image=mask_pil,
                    num_inference_steps=INFERENCE_STEPS,
                    guidance_scale=g,
                    strength=s,  # ⭐ 핵심 — 변형 강도 명시
                    generator=torch.Generator(device).manual_seed(SEED),
                ).images[0]
            results[key] = np.array(output)
            print(f'  완료: {time.time() - t0:.1f}초')
        except TypeError as e:
            # 모델이 strength 미지원 시 fallback
            print(f'  ⚠ strength 미지원, 기본 호출')
            with torch.autocast('cuda'):
                output = pipe(
                    prompt=current['positive'],
                    negative_prompt=current['negative'],
                    image=image_pil,
                    mask_image=mask_pil,
                    num_inference_steps=INFERENCE_STEPS,
                    guidance_scale=g,
                    generator=torch.Generator(device).manual_seed(SEED),
                ).images[0]
            results[key] = np.array(output)

print(f'\n총 소요: {(time.time() - total_start)/60:.1f}분')

## 6. 결과 시각화 — 3×3 Grid

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 20))

axes[0, 0].imshow(image_rgb); axes[0, 0].set_title('원본 (Before)', fontsize=14, fontweight='bold'); axes[0, 0].axis('off')
axes[0, 1].imshow(mask, cmap='gray'); axes[0, 1].set_title(f'Mask (+{ds}px)', fontsize=14); axes[0, 1].axis('off')
axes[0, 2].axis('off')
axes[0, 2].text(0.5, 0.5, f'시술: {PROC_ID}\n\nModel:\n{MODEL_USED.split("/")[-1]}\n\n3 Strength × 3 Guidance',
                ha='center', va='center', fontsize=11, transform=axes[0, 2].transAxes)

for i, s in enumerate(STRENGTH_VALUES):
    for j, g in enumerate(GUIDANCE_VALUES):
        key = f's{s}_g{g}'
        row = i + 1
        axes[row, j].imshow(results[key])
        title = f'strength={s}, g={g}'
        if s == 0.95 and g == 10.0:
            title += '\n⭐ 추천'
        axes[row, j].set_title(title, fontsize=11)
        axes[row, j].axis('off')

plt.suptitle(f'{PROC_ID} — 강력한 Face Inpainting (Strength × Guidance)',
             fontsize=16, fontweight='bold', y=1.0)
plt.tight_layout()
PNG = OUTPUT_DIR / f'{PROC_ID}_strong_grid.png'
plt.savefig(PNG, dpi=120, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n📦 {PNG}')

## 7. Before/After 직접 비교

In [ ]:
BEST_KEY = 's0.95_g10.0'  # 추천

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_rgb); axes[0].set_title('Before (원본)', fontsize=14, fontweight='bold'); axes[0].axis('off')
axes[1].imshow(results['s0.85_g8.5']); axes[1].set_title('약함 (s=0.85, g=8.5)', fontsize=12, color='gray'); axes[1].axis('off')
axes[2].imshow(results[BEST_KEY]); axes[2].set_title(f'⭐ 추천 ({BEST_KEY})\n강한 변형 + 자연스러움', fontsize=12, color='red', fontweight='bold'); axes[2].axis('off')

plt.suptitle(f'{PROC_ID} — Face 특화 모델 강한 변형 결과', fontsize=15, fontweight='bold')
plt.tight_layout()
PNG_BA = OUTPUT_DIR / f'{PROC_ID}_strong_before_after.png'
plt.savefig(PNG_BA, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n📦 {PNG_BA}')

In [ ]:
# 결과 저장 + Zip
import json, imageio, shutil

config = {
    'procedure': PROC_ID,
    'model': MODEL_USED,
    'inference_steps': INFERENCE_STEPS,
    'strength_values': STRENGTH_VALUES,
    'guidance_values': GUIDANCE_VALUES,
    'mask_dilate_px': ds,
    'mask_pct': float(mask_pct),
    'image_size': SIZE,
    'best_combination': BEST_KEY,
    'seed': SEED,
}

with open(OUTPUT_DIR / f'{PROC_ID}_strong_config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

for key, img in results.items():
    imageio.imwrite(OUTPUT_DIR / f'{PROC_ID}_{key}.png', img)

ZIP = OUTPUT_DIR.parent / 'strong_inpaint_results.zip'
shutil.make_archive(str(ZIP).replace('.zip', ''), 'zip', OUTPUT_DIR)
print(f'📥 Zip: {ZIP}')

print('\n=== 사용 모델 ===')
print(f'  {MODEL_USED}')
print(f'\n=== 추천 설정 ===')
print(f'  strength: 0.95 (강한 변형)')
print(f'  guidance: 10.0')
print(f'  mask_dilate: {ds}px (대폭 확장)')
print(f'  prompt: dramatic transformation 키워드 포함')

## ✅ 완료 체크

- [ ] Face 특화 모델 로드 (DreamShaper / RealisticVision / etc.)
- [ ] Mask 대폭 확장 (40px+)
- [ ] Strength 파라미터 명시 (0.85 / 0.95 / 1.0)
- [ ] 강한 prompt (dramatic transformation)
- [ ] 9가지 조합 실험
- [ ] PNG 결과 다운로드

## 🎯 핵심 개선 사항 (vs 노트북 13/20/21)

| 항목 | 노트북 13/20/21 | **노트북 22** |
|------|---------------|-----------|
| Model | runwayml SD | **Face 특화 (DreamShaper, etc.)** |
| Strength | 미지원/미명시 | **명시 (0.85~1.0)** |
| Mask dilate | 8~35px | **40px+ (대폭)** |
| Prompt | identity 강조 | **dramatic transformation 강조** |
| 결과 변형 | 약함 | **강함 + 자연스러움** |